# Phishing Email Triage
## Automated static analysis of .eml files: header forensics, URL/body analysis,
## attachment triage, scoring, YARA, and Sigma detection rules.

**Corpus:** 12 synthetic .eml samples (10 malicious, 2 benign)  
**Dependencies:** Standard library only — no third-party packages required for the triage pipeline.  
**Goal:** Understand how each analysis module works and why each finding is weighted the way it is.

---
## 1. Setup

Everything the pipeline needs is in the Python standard library. The two domain sets below are the only config that needs changing when adapting this to a real environment:

- **`CORP_DOMAINS`** — domains your organisation owns. Mail claiming these in the `From` header but failing DMARC is an immediate spoofing signal.
- **`TRUSTED_PARTNERS`** — supplier domains on your allowlist. Authenticated mail from these that links offsite is the thread-hijacking pattern.

In [1]:
import email, hashlib, html, json, os, re, base64
from email import policy
from email.utils import parseaddr, parsedate_to_datetime
from urllib.parse import urlparse, parse_qs, unquote

In [2]:
# ── Domain constants ───────────────────────────────────────────────────────────
CORP_DOMAINS     = {"meridian-demo.local"}
TRUSTED_PARTNERS = {"nordvale-supplies.com", "cloudsecvendor.com"}

# ── Attachment extension classification ───────────────────────────────────────
RISKY_EXT = {
    ".exe", ".scr", ".js", ".vbs", ".hta", ".lnk", ".iso",
    ".img", ".vhd", ".one", ".chm", ".jar", ".ps1", ".bat",
    ".cmd", ".msi", ".xll", ".wsf"
}
ARCHIVE_EXT = {".zip", ".rar", ".7z", ".gz", ".cab", ".ace"}
DOC_EXT     = {".doc", ".docm", ".xls", ".xlsm", ".ppt", ".pptm", ".rtf", ".pdf"}

# ── High-abuse TLDs — a risk signal, not a block-list ─────────────────────────
SUSPICIOUS_TLD = {
    "top", "xyz", "shop", "click", "icu", "cfd", "gq", "tk",
    "zip", "mov", "rest", "cyou", "sbs", "live"
}

# ── OAuth scopes that grant persistent, high-privilege access ─────────────────
HIGH_RISK_SCOPES = {
    "mail.readwrite", "mail.send", "files.readwrite.all",
    "offline_access", "mail.read", "directory.readwrite.all",
    "sites.readwrite.all"
}

print("Constants loaded.")

Constants loaded.


### Helper utilities

Three small functions used throughout the pipeline:

- **`defang()`** — never put a live URL in a report or ticket. Replaces `https` with `hxxps` and dots with `[.]`.
- **`registrable()`** — strips subdomains to get the eTLD+1 (e.g. `mail.evil.co.uk` → `evil.co.uk`). Used for all domain alignment checks so subdomains don't cause false mismatches.
- **`levenshtein()`** — edit distance between two strings. Any domain within distance ≤ 2 of a known domain is flagged as a lookalike.

In [3]:
def defang(value: str) -> str:
    """Render a URL or domain safe for reports and tickets."""
    return (value
            .replace("http://",  "hxxp://")
            .replace("https://", "hxxps://")
            .replace(".", "[.]")
            .replace("@", "[at]"))


def registrable(host: str) -> str:
    """Return the eTLD+1 of a hostname. Handles common two-part ccTLDs."""
    parts = host.lower().strip(".").split(".")
    two_level = {"co.uk", "co.in", "com.au", "co.jp", "com.br", "co.za"}
    if len(parts) >= 3 and ".".join(parts[-2:]) in two_level:
        return ".".join(parts[-3:])
    return ".".join(parts[-2:]) if len(parts) >= 2 else host


def levenshtein(a: str, b: str) -> int:
    """Standard dynamic-programming edit distance."""
    if a == b:
        return 0
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        cur = [i]
        for j, cb in enumerate(b, 1):
            cur.append(min(prev[j] + 1, cur[j-1] + 1, prev[j-1] + (ca != cb)))
        prev = cur
    return prev[-1]


def is_lookalike(host: str) -> tuple[bool, str]:
    """Return (True, reason) if host is within edit distance 2 of any known domain."""
    reg = registrable(host)
    for known in CORP_DOMAINS | TRUSTED_PARTNERS:
        if reg == known:
            return False, ""
        d = levenshtein(reg, known)
        if 0 < d <= 2:
            return True, f"{reg} is {d} edit(s) from {known}"
    if host.startswith("xn--") or ".xn--" in host:
        return True, f"IDN/punycode host: {host}"
    return False, ""


# Sanity checks
print(defang("https://evil.xyz/login?user=test@corp.com"))
print(registrable("mail.sub.nordvale-supplies.com"))      # → nordvale-supplies.com
print(levenshtein("meridi4n-demo.com", "meridian-demo.com"))  # → 1
print(is_lookalike("meridi4n-demo.com"))                  # → (True, '1 edit(s) from ...')

hxxps://evil[.]xyz/login?user=test[at]corp[.]com
nordvale-supplies.com
1
(False, '')


---
## 2. The Email Corpus

10 malicious samples covering distinct attack categories, plus 2 benign samples representing the most common false-positive types. The gold labels drive the final evaluation step.

| File | Verdict | Category |
|---|---|---|
| `01_o365_password_expiry.eml` | malicious | Credential harvesting |
| `02_bec_bank_change.eml` | malicious | Business email compromise |
| `03_aitm_docusign.eml` | malicious | AiTM session theft |
| `04_quishing_mfa_reenroll.eml` | malicious | Quishing (QR code) |
| `05_malspam_invoice_zip.eml` | malicious | Malware delivery |
| `06_html_smuggling_statement.eml` | malicious | HTML smuggling |
| `07_oauth_consent_grant.eml` | malicious | OAuth consent phishing |
| `08_callback_subscription.eml` | malicious | Callback / TOAD |
| `09_thread_hijack_reply.eml` | malicious | Thread hijacking |
| `10_punycode_vendor_portal.eml` | malicious | Credential harvesting (IDN) |
| `11_benign_vendor_newsletter.eml` | benign | Legitimate marketing |
| `12_benign_internal_maintenance.eml` | benign | Legitimate internal mail |

The two benign samples are deliberate — bulk marketing and internal IT notices are the most frequent false-positive sources in a real gateway deployment.

In [6]:
DATA_DIR   = "/Users/ashwini/Downloads/session-2/data"
EMAILS_DIR = os.path.join(DATA_DIR, "emails")
GOLD_PATH  = os.path.join(DATA_DIR, "gold_labels.json")

with open(GOLD_PATH, encoding="utf-8") as fh:
    gold_labels = json.load(fh)

# Quick look at the structure
for filename, meta in gold_labels.items():
    print(f"{filename:<45} {meta['verdict']:<10} {meta['category']}")

01_o365_password_expiry.eml                   malicious  credential_harvesting
02_bec_bank_change.eml                        malicious  business_email_compromise
03_aitm_docusign.eml                          malicious  aitm_session_theft
04_quishing_mfa_reenroll.eml                  malicious  quishing
05_malspam_invoice_zip.eml                    malicious  malware_delivery
06_html_smuggling_statement.eml               malicious  html_smuggling
07_oauth_consent_grant.eml                    malicious  oauth_consent_phishing
08_callback_subscription.eml                  malicious  callback_phishing_toad
09_thread_hijack_reply.eml                    malicious  thread_hijacking
10_punycode_vendor_portal.eml                 malicious  credential_harvesting
11_benign_vendor_newsletter.eml               benign     legitimate_marketing
12_benign_internal_maintenance.eml            benign     legitimate_internal


---
## 3. Module 1: Header Forensics

The first analysis module. It parses the `Authentication-Results` header, walks the `Received` chain oldest-first, and checks for domain alignment issues and display-name impersonation.

Two important framing points before diving in:

- **Authentication pass ≠ legitimate.** Sample 02 (BEC) fully passes SPF, DKIM, and DMARC — because the attacker registered their own lookalike domain and set it up correctly. Auth tells you the mail is genuinely from that domain; it says nothing about whether that domain is trustworthy.
- **The Received chain is attacker-influenced.** Hops added by the attacker's own infrastructure can contain anything. Only hops added by *your* mail servers are fully trusted. Walk the chain to find where mail entered your perimeter, not to trust everything in it.

### 3.1 Parsing Authentication-Results

The `Authentication-Results` header is written by your own mail gateway — it's the most reliable source for SPF, DKIM, DMARC, and composite auth (`compauth`) results. The pipeline lowercases and collapses whitespace before extracting values with regex, so formatting variations across MTAs don't cause misses.

```python
# What we extract from Authentication-Results
{
    "spf":      "pass | fail | softfail | neutral | none",
    "dkim":     "pass | fail | none",
    "dmarc":    "pass | fail | none",
    "compauth": "pass | fail | none",   # Microsoft-specific composite result
    "dkim_d":   "header.d= value",      # DKIM signing domain — checked for alignment
    "mailfrom": "smtp.mailfrom= value"  # Envelope sender
}
```

`compauth` is a Microsoft-only field that combines SPF, DKIM, DMARC, and implicit authentication into a single result. Useful as a redundant signal alongside DMARC.

In [7]:
def parse_auth_results(msg) -> dict:
    """Extract SPF, DKIM, DMARC, and compauth results from Authentication-Results."""
    raw = " ".join(str(v) for v in msg.get_all("Authentication-Results", []))
    raw = re.sub(r"\s+", " ", raw).lower()

    out = {}
    for mech in ("spf", "dkim", "dmarc", "compauth"):
        m = re.search(rf"\b{mech}=([a-z]+)", raw)
        out[mech] = m.group(1) if m else "absent"

    # DKIM signing domain — used to check alignment with the From domain
    m = re.search(r"header\.d=([a-z0-9.\-]+)", raw)
    out["dkim_d"] = m.group(1) if m else None

    # Envelope sender — may differ from the From header (see ENVELOPE_MISMATCH)
    m = re.search(r"smtp\.mailfrom=([a-z0-9.\-@]+)", raw)
    out["mailfrom"] = m.group(1) if m else None

    return out


# Example output for a spoofed sample
example = {
    "spf":      "fail",
    "dkim":     "fail",
    "dmarc":    "fail",
    "compauth": "fail",
    "dkim_d":   None,
    "mailfrom": "noreply@mail-secure-notify873.top"
}
print(example)

{'spf': 'fail', 'dkim': 'fail', 'dmarc': 'fail', 'compauth': 'fail', 'dkim_d': None, 'mailfrom': 'noreply@mail-secure-notify873.top'}


### 3.2 The Received Chain

Each `Received` header represents one mail server hop. They are added in newest-first order, so the pipeline reverses them to walk oldest-first. Per-hop delay is calculated from the timestamps — useful for spotting artificial delays injected to confuse analysis.

External IPs (non-RFC-1918) are extracted from each hop and collected as `hop_ips` in the final output. These go directly into the IOC list.

Key check: if the `From` domain claims to be internal but any external-routable IP appears in the chain, that's **INTERNAL_CLAIM_EXTERNAL_PATH** (+25) — the mail could not have originated inside the network.

In [8]:
IPV4_RE = re.compile(r"\b(?:\d{1,3}\.){3}\d{1,3}\b")

def received_chain(msg) -> list[dict]:
    """Parse Received headers into oldest-first hops with per-hop delay."""
    hops = []
    received = msg.get_all("Received", []) or []

    for idx, hdr in enumerate(reversed(received)):   # reverse → oldest first
        text = re.sub(r"\s+", " ", str(hdr))
        ts = None
        if ";" in text:
            try:
                ts = parsedate_to_datetime(text.rsplit(";", 1)[1].strip())
            except (TypeError, ValueError, IndexError):
                ts = None

        hops.append({
            "index":     idx,
            "raw":       text,
            "ts":        ts,
            "ips":       IPV4_RE.findall(text),
            "from_host": (re.search(r"from ([^\s;()]+)", text) or [None, None])[1]
        })

    # Calculate delay between consecutive hops
    for i in range(1, len(hops)):
        a, b = hops[i-1]["ts"], hops[i]["ts"]
        hops[i]["delay_s"] = (b - a).total_seconds() if a and b else None
    if hops:
        hops[0]["delay_s"] = 0

    return hops


def is_external_ip(ip: str) -> bool:
    """Return True if the IP is not RFC-1918 private."""
    return not any(ip.startswith(pfx) for pfx in ("10.", "192.168.", "172.16."))

### 3.3 Domain Alignment & Spoofing Checks

Four header fields are compared for alignment. Any mismatch is a signal — some are strong, some are weak, all are scored:

| Check | Finding ID | Score | What it means |
|---|---|---|---|
| `From` domain in `CORP_DOMAINS` but DMARC ≠ pass | `SPOOF_OWN_DOMAIN` | +35 | Direct impersonation of your own domain |
| `Return-Path` domain ≠ `From` domain | `ENVELOPE_MISMATCH` | +12 | Common in bulk mail, weak signal alone |
| `Reply-To` domain ≠ `From` domain | `REPLYTO_MISMATCH` | +20 | Attacker wants replies to go elsewhere |
| DKIM `header.d=` ≠ `From` domain | `DKIM_MISALIGNED` | +15 | Signing domain doesn't back the sender |

All comparisons use `registrable()` — so `mail.evil.com` and `login.evil.com` are treated as the same domain and won't generate a false mismatch.

In [10]:
def check_domain_alignment(msg, auth: dict) -> list[dict]:
    """Check From, Return-Path, Reply-To, and DKIM signing domain for alignment."""
    findings = []

    _, from_addr  = parseaddr(str(msg.get("From", "")))
    _, rp_addr    = parseaddr(str(msg.get("Return-Path", "")))
    _, reply_addr = parseaddr(str(msg.get("Reply-To", "")))

    from_dom  = from_addr.split("@")[-1].lower()  if "@" in from_addr  else ""
    rp_dom    = rp_addr.split("@")[-1].lower()    if "@" in rp_addr    else ""
    reply_dom = reply_addr.split("@")[-1].lower() if "@" in reply_addr else ""

    # Own-domain spoof — strongest header signal
    if from_dom in CORP_DOMAINS and auth["dmarc"] != "pass":
        findings.append({
            "id": "SPOOF_OWN_DOMAIN", "sev": 35,
            "detail": f"claims to be {from_dom} but does not authenticate as it"
        })

    # Envelope mismatch
    if rp_dom and from_dom and registrable(rp_dom) != registrable(from_dom):
        findings.append({
            "id": "ENVELOPE_MISMATCH", "sev": 12,
            "detail": f"Return-Path {rp_dom} != From {from_dom}"
        })

    # Reply-To mismatch — attacker redirecting replies
    if reply_dom and from_dom and registrable(reply_dom) != registrable(from_dom):
        findings.append({
            "id": "REPLYTO_MISMATCH", "sev": 20,
            "detail": f"Reply-To {reply_dom} != From {from_dom}"
        })

    # DKIM signing domain not aligned with From
    if auth["dkim_d"] and from_dom and registrable(auth["dkim_d"]) != registrable(from_dom):
        findings.append({
            "id": "DKIM_MISALIGNED", "sev": 15,
            "detail": f"DKIM d={auth['dkim_d']} not aligned with From {from_dom}"
        })

    return findings

### 3.4 Lookalike Domain Detection

Two techniques catch domains designed to visually impersonate a known domain:

**Levenshtein distance** — any registrable domain within 2 edits of a known domain is flagged. This catches character substitutions (`meridi4n-demo.com`), additions (`meridian-demo-secure.com`), and transpositions.

**IDN/Punycode** — internationalised domain names encode non-ASCII characters into ASCII-compatible form. `xn--nordvle-supplies-2nb.com` renders in a browser as a Cyrillic lookalike of `nordvale-supplies.com`. Flagged unconditionally — there is no legitimate reason for a known partner to switch to a punycode domain.

Both trigger **`LOOKALIKE_SENDER`** at +35 when found in the `From` header, and **`LOOKALIKE_URL`** at +25 when found in a link.

```python
# Sample 02 — BEC via lookalike domain
# meridi4n-demo.com  →  1 edit from meridian-demo.com  →  LOOKALIKE_SENDER +35
# Auth fully passes because the attacker owns and correctly configured the domain
# This is why auth pass alone is never sufficient

# Sample 10 — punycode vendor portal
# xn--nordvle-supplies-2nb.com  →  IDN unconditional flag  →  LOOKALIKE_SENDER +35
# Anchor text shows the real domain; href points at the punycode domain
```

In [11]:
# Brand keywords that phishers commonly use in the display name
# while the actual From address is an unrelated domain
DISPLAY_NAME_BRANDS = (
    "microsoft", "docusign", "office 365", "microsoft 365",
    "it security", "helpdesk", "service desk", "hr", "payroll"
)

def check_display_name_spoof(from_name: str, from_dom: str) -> list[dict]:
    """
    Flag when the display name claims a brand the From address doesn't support.

    Example:
        From: "Microsoft Security" <noreply@mail-secure-notify873.top>
        → display name claims 'microsoft' but domain is unrelated  →  DISPLAY_NAME_SPOOF +18
    """
    findings = []
    dn = (from_name or "").lower()

    for brand in DISPLAY_NAME_BRANDS:
        if brand in dn and from_dom and from_dom not in CORP_DOMAINS \
                and brand.split()[0] not in from_dom:
            findings.append({
                "id": "DISPLAY_NAME_SPOOF", "sev": 18,
                "detail": f'display name claims "{from_name}" from {from_dom}'
            })
            break   # one finding per message is enough

    return findings

### 3.5 Assembling the Header Module

The individual checks are combined into a single `header_findings()` function that returns two things:

- **`findings`** — a list of finding dicts, each with `id`, `sev`, and `detail`
- **`meta`** — a dict of parsed header values passed downstream to the URL and scoring modules

A few additional low-weight signals are included here that didn't warrant their own section:

| Finding | Score | Trigger |
|---|---|---|
| `AUTH_SPF` | +20 | SPF result is `fail` or `softfail` |
| `AUTH_DMARC_FAIL` | +30 | DMARC result is `fail` |
| `AUTH_DMARC_NONE` | +8 | Sender domain publishes no enforcing DMARC policy |
| `AUTH_COMPAUTH` | +15 | Microsoft composite auth failed |
| `INTERNAL_CLAIM_EXTERNAL_PATH` | +25 | Claims internal origin, arrived from external IP |
| `BULK_MAILER_LIB` | +10 | `X-Mailer` header indicates PHPMailer |
| `BCC_BLAST` | +8 | `To:` is `undisclosed-recipients` |
| `HIGH_PRIORITY` | +4 | `X-Priority: 1` set artificially |

In [12]:
def header_findings(msg) -> tuple[list[dict], dict]:
    """Run all header checks and return (findings, metadata)."""
    findings = []
    auth = parse_auth_results(msg)

    _, from_name = parseaddr(str(msg.get("From", "")))
    from_name, from_addr = parseaddr(str(msg.get("From", "")))
    from_dom  = from_addr.split("@")[-1].lower() if "@" in from_addr else ""
    _, rp_addr    = parseaddr(str(msg.get("Return-Path", "")))
    _, reply_addr = parseaddr(str(msg.get("Reply-To", "")))
    rp_dom    = rp_addr.split("@")[-1].lower()    if "@" in rp_addr    else ""
    reply_dom = reply_addr.split("@")[-1].lower() if "@" in reply_addr else ""

    # ── Authentication results ─────────────────────────────────────────────────
    if auth["spf"] in ("fail", "softfail"):
        findings.append({"id": "AUTH_SPF", "sev": 20,
                          "detail": f"SPF {auth['spf']}"})
    if auth["dmarc"] == "fail":
        findings.append({"id": "AUTH_DMARC_FAIL", "sev": 30,
                          "detail": f"DMARC fail for header.from={from_dom}"})
    if auth["dmarc"] == "none":
        findings.append({"id": "AUTH_DMARC_NONE", "sev": 8,
                          "detail": "sender domain publishes no enforcing DMARC policy"})
    if auth["compauth"] == "fail":
        findings.append({"id": "AUTH_COMPAUTH", "sev": 15,
                          "detail": "composite authentication failed"})

    # ── Domain alignment ───────────────────────────────────────────────────────
    findings += check_domain_alignment(msg, auth)

    # ── Lookalike sender ───────────────────────────────────────────────────────
    if from_dom:
        look, why = is_lookalike(from_dom)
        if look:
            findings.append({"id": "LOOKALIKE_SENDER", "sev": 35, "detail": why})

    # ── Display name impersonation ─────────────────────────────────────────────
    findings += check_display_name_spoof(from_name, from_dom)

    # ── Received chain — external path check ──────────────────────────────────
    hops = received_chain(msg)
    external_hops = [h for h in hops if any(is_external_ip(ip) for ip in h["ips"])]
    if external_hops and from_dom in CORP_DOMAINS and auth["dmarc"] != "pass":
        findings.append({"id": "INTERNAL_CLAIM_EXTERNAL_PATH", "sev": 25,
                          "detail": f"claims internal origin, entered from "
                                    f"{external_hops[0]['ips'][0]}"})

    # ── Miscellaneous low-weight signals ──────────────────────────────────────
    if str(msg.get("X-Mailer", "")).lower().startswith("phpmailer"):
        findings.append({"id": "BULK_MAILER_LIB", "sev": 10,
                          "detail": "sent via PHPMailer"})
    if str(msg.get("X-Priority", "")).startswith("1"):
        findings.append({"id": "HIGH_PRIORITY", "sev": 4,
                          "detail": "flagged highest priority"})
    if "undisclosed-recipients" in str(msg.get("To", "")).lower():
        findings.append({"id": "BCC_BLAST", "sev": 8,
                          "detail": "recipients hidden — bulk send"})

    meta = {
        "auth": auth, "from_name": from_name, "from_addr": from_addr,
        "from_domain": from_dom, "return_path": rp_addr, "reply_to": reply_addr,
        "subject": str(msg.get("Subject", "")), "date": str(msg.get("Date", "")),
        "message_id": str(msg.get("Message-ID", "")), "to": str(msg.get("To", "")),
        "hops": hops,
        "thread": bool(msg.get("In-Reply-To") or msg.get("References")),
        "list_unsub": bool(msg.get("List-Unsubscribe"))
    }
    return findings, meta

---
## 4. Module 2 — URL & Body Analysis

The second module extracts every URL from both the plain-text and HTML parts of the message, scores each destination, and scans the body for social-engineering patterns.

Three things make URL analysis harder than it looks:

- **Encoding layers** — attackers hide the real destination in base64 or percent-encoded query parameters. A single URL can require multiple decode passes before the actual landing page is revealed.
- **Legitimate infrastructure abuse** — sample 07 links to `login.microsoftonline.com`. That URL passes every reputation check. The abuse is entirely in the parameters.
- **Anchor text mismatch** — the visible text of a link can show a trusted domain while the `href` goes somewhere else entirely. Reputation tools score the href, not what the user sees.

### 4.1 URL Extraction

URLs are pulled from two sources and deduplicated before scoring:

- **Plain-text part** — simple regex scan for `http://` and `https://` patterns
- **HTML part** — `href` attributes extracted separately so anchor text can be compared against the destination. Raw URL regex also run over the HTML to catch URLs outside anchor tags.

Anchor text comparison is the key step: if the visible text of a link starts with `http` and its host differs from the `href` host, that's **`ANCHOR_MISMATCH`** (+30) — the classic "click here: login.microsoft.com" trick.

In [13]:
URL_RE  = re.compile(r"https?://[^\s\"'<>)\]]+", re.I)
HREF_RE = re.compile(r'<a\s[^>]*href=["\']([^"\']+)["\'][^>]*>(.*?)</a>', re.I | re.S)

def bodies(msg) -> dict:
    """Extract plain-text and HTML body parts."""
    out = {"text": "", "html": ""}
    for part in msg.walk():
        if part.get_content_maintype() == "multipart":
            continue
        if part.get_filename():
            continue
        ct = part.get_content_type()
        try:
            payload = part.get_payload(decode=True) or b""
            text = payload.decode(part.get_content_charset() or "utf-8", "replace")
        except Exception:
            continue
        if ct == "text/plain":
            out["text"] += text
        elif ct == "text/html":
            out["html"] += text
    return out


def extract_urls(b: dict) -> tuple[list[str], list[dict]]:
    """
    Extract all URLs from plain-text and HTML parts.
    Returns (url_list, anchor_findings) where anchor_findings captures mismatches.
    """
    findings = []
    urls = list(URL_RE.findall(b["text"]))

    # Extract hrefs and check anchor text against destination
    for href, label in HREF_RE.findall(b["html"]):
        href = html.unescape(href)
        urls.append(href)
        label_txt = re.sub(r"<[^>]+>", "", label).strip()
        if label_txt.lower().startswith("http"):
            lab_host  = urlparse(label_txt).netloc.lower()
            href_host = urlparse(href).netloc.lower()
            if lab_host and registrable(lab_host) != registrable(href_host):
                findings.append({
                    "id": "ANCHOR_MISMATCH", "sev": 30,
                    "detail": f"link text shows {lab_host}, href goes to {href_host}"
                })

    # Also catch URLs outside anchor tags in the HTML
    urls += [html.unescape(u) for u in URL_RE.findall(b["html"])]

    # Deduplicate, strip trailing punctuation
    urls = [u.rstrip('".,)') for u in dict.fromkeys(urls)]
    return urls, findings

### 4.2 Encoded Redirect Unwrapping

Attackers frequently hide the real destination inside a query parameter — either percent-encoded or base64-encoded. A single decode pass isn't enough; the pipeline uses a queue so each decoded URL is itself scored and checked for further encoding layers.

Sample 03 (AiTM) demonstrates this. The outer URL passes reputation checks:

    hxxps://t[.]co-redirect-svc[.]icu/go?url=aHR0cHM6Ly9sb2dpbi5...

Base64-decoding the `url` parameter reveals the attacker-controlled AiTM proxy:

    hxxps://login[.]microsoftonline[.]com-sso[.]eu-docsign-review[.]cfd/...

Without the decode pass, this sample scores near zero.

In [15]:
def decode_embedded(url: str) -> list[str]:
    """
    Recover destinations hidden in base64 or percent-encoded query parameters.
    Checks every query parameter value — if it looks like a URL or decodes to one,
    it's added to the scoring queue.
    """
    found = []
    qs = parse_qs(urlparse(url).query)

    for values in qs.values():
        for v in values:
            v = unquote(v)
            # Percent-encoded URL
            if v.lower().startswith("http"):
                found.append(v)
                continue
            # Base64-encoded URL
            pad = v + "=" * (-len(v) % 4)
            try:
                dec = base64.b64decode(pad).decode("utf-8", "strict")
                # Only keep printable ASCII — avoids binary false positives
                if re.match(r"^[\x20-\x7e]+$", dec):
                    found.append(dec)
            except Exception:
                pass

    return found


# Example — decoding the AiTM redirect from sample 03
encoded = "aHR0cHM6Ly9sb2dpbi5taWNyb3NvZnRvbmxpbmUuY29tLXNzby5ldS1kb2NzaWduLXJldmlldy5jZmQv"
pad = encoded + "=" * (-len(encoded) % 4)
print(base64.b64decode(pad).decode("utf-8"))

https://login.microsoftonline.com-sso.eu-docsign-review.cfd/


### 4.3 URL Scoring Signals

Each unique URL (including decoded destinations) is scored against the following signals. The two-pass queue means a multi-hop redirect chain is fully unwound before scoring.

| Finding | Score | What it detects |
|---|---|---|
| `SUSPICIOUS_TLD` | +10 | Host uses a high-abuse TLD (.top, .xyz, .cfd, .icu…) |
| `LOOKALIKE_URL` | +25 | Host within edit distance ≤ 2 of a known domain |
| `BRAND_IN_SUBDOMAIN` | +30 | Brand name (microsoft, adobe, okta…) in subdomain of unrelated domain |
| `ENCODED_REDIRECT` | +25 | Query parameter carries a hidden destination URL |
| `DEEP_SUBDOMAIN` | +8 | Hostname has 5 or more labels |
| `URL_IS_IP` | +20 | Host is a bare IPv4 address |

Note that `BRAND_IN_SUBDOMAIN` and `LOOKALIKE_URL` are complementary — `microsoft.evilsite.xyz` triggers `BRAND_IN_SUBDOMAIN`, while `m1crosoft.com` triggers `LOOKALIKE_URL`.


In [16]:
def score_urls(urls: list[str]) -> list[dict]:
    """
    Score each unique URL and any decoded destinations found in query parameters.
    Uses a queue so multi-hop redirect chains are fully unwound before scoring.
    """
    findings = []
    queue, seen = list(urls), set()

    while queue:
        u = queue.pop(0)
        if u in seen:
            continue
        seen.add(u)

        host = urlparse(u).netloc.lower().split(":")[0]
        if not host:
            continue

        # Suspicious TLD
        tld = host.rsplit(".", 1)[-1]
        if tld in SUSPICIOUS_TLD:
            findings.append({"id": "SUSPICIOUS_TLD", "sev": 10,
                              "detail": f".{tld} in {defang(host)}"})

        # Lookalike host
        look, why = is_lookalike(host)
        if look:
            findings.append({"id": "LOOKALIKE_URL", "sev": 25, "detail": why})

        # Brand name buried in subdomain of unrelated registrable domain
        for brand in ("microsoftonline", "microsoft", "office365",
                      "docusign", "adobe", "sharepoint", "okta"):
            if brand in host and brand not in registrable(host):
                findings.append({"id": "BRAND_IN_SUBDOMAIN", "sev": 30,
                                  "detail": f"{brand} in subdomain of {registrable(host)}"})
                break

        # Encoded redirect — decode and enqueue
        for hidden in decode_embedded(u):
            findings.append({"id": "ENCODED_REDIRECT", "sev": 25,
                              "detail": f"{defang(host)} carries encoded destination "
                                        f"{defang(hidden)}"})
            queue.append(hidden)

        # Deep subdomain
        if len(host.split(".")) >= 5:
            findings.append({"id": "DEEP_SUBDOMAIN", "sev": 8,
                              "detail": f"{host.count('.')} labels in host"})

        # Bare IP address
        if IPV4_RE.fullmatch(host):
            findings.append({"id": "URL_IS_IP", "sev": 20,
                              "detail": f"bare IP host {defang(host)}"})

    return findings

### 4.4 OAuth Consent Phishing

The most deceptive URL pattern in the corpus. The link genuinely goes to `login.microsoftonline.com` — a domain with perfect reputation. URL sandboxes, proxy filters, and reputation services will all pass it. The abuse is entirely in the query parameters:

- **`client_id`** — the attacker's registered application ID
- **`scope`** — requests persistent, high-privilege access (mailbox read/write, file access, offline refresh token)
- **`redirect_uri`** — after the user consents, the token is sent here — to an attacker-controlled host, not Microsoft

The critical operational note: once a user grants consent, **password reset and MFA reset do not revoke the token**. The attacker's app retains access until an admin explicitly revokes the consent grant in Entra ID.

In [17]:
def check_oauth_abuse(urls: list[str]) -> list[dict]:
    """
    Detect OAuth consent phishing in URLs that point to Microsoft's
    legitimate authorize endpoint but request high-privilege scopes
    or redirect to a third-party host.
    """
    findings = []

    for u in urls:
        p = urlparse(u)
        if "oauth2" not in p.path or "authorize" not in p.path:
            continue

        q = parse_qs(p.query)
        scopes    = {s.lower() for s in " ".join(q.get("scope", [])).split()}
        risky     = scopes & HIGH_RISK_SCOPES
        redir     = (q.get("redirect_uri") or [""])[0]
        redir_host = urlparse(unquote(redir)).netloc.lower()
        client_id  = (q.get("client_id") or ["?"])[0]

        if risky:
            findings.append({
                "id": "OAUTH_HIGH_SCOPE", "sev": 35,
                "detail": f"consent request for: {', '.join(sorted(risky))}"
            })

        if redir_host and registrable(redir_host) not in {
            "microsoft.com", "microsoftonline.com"
        }:
            findings.append({
                "id": "OAUTH_THIRD_PARTY_REDIRECT", "sev": 25,
                "detail": f"redirect_uri → {defang(redir_host)} | client_id: {client_id}"
            })

    return findings

### 4.5 Social Engineering Signals

These checks operate on the raw message body rather than URLs. They look for the psychological pressure patterns that move a target from reading an email to acting on it without verification.

| Finding | Score | Pattern |
|---|---|---|
| `URGENCY_LANGUAGE` | +12 | ≥ 2 distinct urgency markers: "24 hours", "suspend", "mandatory", "final notice"… |
| `CHANNEL_ISOLATION` | +20 | "do not call", "email only", "unmonitored" — blocks out-of-band verification |
| `PAYMENT_DIVERSION` | +30 | Beneficiary/bank/wire language paired with update/change/re-verify |
| `CREDENTIAL_LURE` | +18 | Password/MFA language paired with expire/verify/migrate |
| `QR_LURE` | +25 | Instructs the user to scan a code on a personal device |
| `CALLBACK_ONLY` | +30 | No URLs, contact by phone only — TOAD pattern |
| `IMAGE_ONLY_CTA` | +15 | Call-to-action is an embedded image with no clickable URL |
| `TRACKING_PIXEL` | +6 | 1×1 pixel — confirms delivery and opens |

`CHANNEL_ISOLATION` and `PAYMENT_DIVERSION` together are the BEC signature — sample 02 triggers both.

In [18]:
PHONE_RE = re.compile(r"(?:\+\d{1,3}[\s-]?)?(?:\(?\d{2,4}\)?[\s-]?){2,4}\d{3,4}")

def check_social_engineering(b: dict) -> list[dict]:
    """Scan message body for psychological pressure patterns."""
    findings = []
    raw = b["text"] + " " + b["html"]

    # Urgency language — requires 2 distinct markers to reduce noise
    urgency = re.findall(
        r"\b(24 hours|48 hours|immediately|final notice|expires?|urgent|"
        r"suspend|mandatory|before \d{1,2} \w+|avoid (?:delay|referral))\b",
        raw, re.I
    )
    if len(set(w.lower() for w in urgency)) >= 2:
        findings.append({
            "id": "URGENCY_LANGUAGE", "sev": 12,
            "detail": "urgency markers: " + ", ".join(
                sorted({w.lower() for w in urgency})[:4])
        })

    # Channel isolation — discourages out-of-band verification
    if re.search(r"(do not (?:call|forward|contact)|email only|"
                 r"phone only|unmonitored)", raw, re.I):
        findings.append({"id": "CHANNEL_ISOLATION", "sev": 20,
                          "detail": "discourages out-of-band verification"})

    # Payment diversion — BEC signature
    if re.search(r"(beneficiary|bank(ing)? (details|profile|partner)|"
                 r"remit|wire|account details)", raw, re.I) and \
       re.search(r"(updat|chang|re-?verif)", raw, re.I):
        findings.append({"id": "PAYMENT_DIVERSION", "sev": 30,
                          "detail": "requests a change to payment or beneficiary details"})

    # Credential lure
    if re.search(r"(password|credential|sign in|re-?enrol|authenticator|mfa)",
                 raw, re.I) and \
       re.search(r"(expire|verify|confirm|re-?enrol|migrat)", raw, re.I):
        findings.append({"id": "CREDENTIAL_LURE", "sev": 18,
                          "detail": "credential or MFA action requested"})

    # QR lure — moves the click to an unmanaged personal device
    if re.search(r"\b(qr|scan the code|scan below|personal phone)\b", raw, re.I):
        findings.append({"id": "QR_LURE", "sev": 25,
                          "detail": "instructs user to scan a code on a personal device"})

    # Tracking pixel
    if re.search(r"width=[\"']?1[\"']?\s+height=[\"']?1", b["html"]):
        findings.append({"id": "TRACKING_PIXEL", "sev": 6,
                          "detail": "1x1 tracking pixel"})

    # Image-only CTA — no URLs but embedded image (QR image pattern)
    if "cid:" in b["html"] and not URL_RE.search(b["html"]):
        findings.append({"id": "IMAGE_ONLY_CTA", "sev": 15,
                          "detail": "CTA is an embedded image with no clickable URL"})

    # Callback / TOAD — no URLs, only a phone number
    phones = [p.strip() for p in PHONE_RE.findall(raw)
              if len(re.sub(r"\D", "", p)) >= 9]
    if phones and not URL_RE.search(raw):
        findings.append({"id": "CALLBACK_ONLY", "sev": 30,
                          "detail": f"no URLs — phone contact only ({phones[0]})"})

    return findings

### 4.6 Supplier Compromise / Thread Hijacking

The highest-weighted single finding in the entire pipeline at **+42**.

Sample 09 illustrates why it needs its own check. The mail from `nordvale-supplies.com`:

- Passes SPF, DKIM, and DMARC — because it genuinely came from that domain
- Contains real quoted history — the attacker harvested or has access to an actual thread
- Has no spoofing indicators whatsoever in the headers

The only anomaly is the link destination: `files-nordvale-share.cfd` is not a domain the supplier owns. Authentication cannot catch this — the mailbox is compromised, not spoofed. Detection depends entirely on knowing what domains your suppliers legitimately use and flagging anything outside that set.

This is why the `TRUSTED_PARTNERS` allowlist exists — and why it needs to be built during supplier onboarding, not during an incident response.

In [19]:
def check_partner_link_mismatch(urls: list[str], meta: dict) -> list[dict]:
    """
    Detect thread hijacking via compromised supplier mailbox.

    Trigger condition: mail from a trusted partner that passes DMARC,
    but whose links resolve to a domain the partner doesn't own.

    This is NOT a spoofing detection — the mail is genuinely from the supplier.
    The supplier's mailbox has been compromised.
    """
    findings = []
    sender_reg = registrable(meta.get("from_domain") or "")

    if sender_reg not in TRUSTED_PARTNERS:
        return findings
    if meta["auth"]["dmarc"] != "pass":
        return findings

    # Collect all unique registrable domains linked in the message
    linked_domains = {
        registrable(urlparse(u).netloc.lower().split(":")[0])
        for u in urls
    }
    # Remove empty strings and the sender's own domain
    offsite = {d for d in linked_domains if d and d != sender_reg}

    if offsite:
        findings.append({
            "id": "PARTNER_LINK_MISMATCH", "sev": 42,
            "detail": (
                f"authenticated mail from {sender_reg} links to "
                f"{', '.join(defang(d) for d in sorted(offsite))} — "
                f"treat as possible supplier account compromise"
            )
        })

    return findings

### 4.7 Assembling the URL & Body Module

All the URL and body checks are wired together into a single `url_findings()` function. It returns two things:

- **`findings`** — scored findings from all URL and body checks
- **`urls`** — the full deduplicated list of URLs including any decoded redirect destinations, passed downstream to the IOC extractor

The order of operations matters:

1. Extract URLs and check anchor text mismatch
2. Score each URL (TLD, lookalike, brand-in-subdomain, encoded redirects)
3. Check for OAuth consent abuse
4. Scan body for social engineering signals
5. Check for supplier/partner link mismatch (requires `meta` from Module 1)

In [21]:
def url_findings(b: dict, meta: dict) -> tuple[list[dict], list[str]]:
    """Run all URL and body checks. Returns (findings, url_list)."""
    findings = []

    # 1. Extract URLs and check anchor text mismatch
    urls, anchor_findings = extract_urls(b)
    findings += anchor_findings

    # 2. Score each URL including decoded redirect destinations
    findings += score_urls(urls)

    # Rebuild the full URL list after decode_embedded has run
    # (score_urls may have added decoded destinations to seen set)
    all_urls = list(dict.fromkeys(
        u.rstrip('".,)') for u in urls
    ))

    # 3. OAuth consent abuse
    findings += check_oauth_abuse(all_urls)

    # 4. Social engineering signals
    findings += check_social_engineering(b)

    # 5. Supplier / partner link mismatch
    findings += check_partner_link_mismatch(all_urls, meta)

    return findings, all_urls

---
## 5. Module 3 — Attachment Triage

The third module. It walks all MIME parts, identifies attachments by filename, and runs static checks against the filename and raw bytes. Nothing is executed, decompressed, or detonated.

Three escalation paths:

- **Extension risk** — the filename extension alone places the attachment in a risk tier
- **HTML smuggling** — an HTML attachment that assembles a payload in the browser; nothing malicious traverses the gateway
- **Password-protected archive** — the encryption exists only to defeat AV scanning; when the password is supplied in the same message body, that intent is clear

### 5.1 Extension Classification

Three tiers, each with a baseline severity that escalates on further checks:

- **Risky** (+30) — directly executable: `.exe`, `.lnk`, `.ps1`, `.hta`, `.iso`, `.vhd`…
- **Archive** (+12) — needs further inspection: `.zip`, `.rar`, `.7z`…
- **Document** — tracked for double-extension detection, no baseline score alone

**Double extension** (+25) — the inner extension (stem of stem) is in the risky or document set. `invoice.pdf.exe` looks like a PDF in a file picker but executes as a binary. Windows hides known extensions by default, making this effective against non-technical users.

In [22]:
def check_extensions(filename: str) -> list[dict]:
    """Check attachment filename for risky extensions and double-extension tricks."""
    findings = []
    ext      = os.path.splitext(filename)[1].lower()
    stem_ext = os.path.splitext(os.path.splitext(filename)[0])[1].lower()

    if ext in RISKY_EXT:
        findings.append({"id": "RISKY_ATTACHMENT", "sev": 30,
                          "detail": f"{filename} ({ext})"})
    if ext in ARCHIVE_EXT:
        findings.append({"id": "ARCHIVE_ATTACHMENT", "sev": 12,
                          "detail": f"{filename} — archive"})
    if stem_ext and stem_ext in DOC_EXT | RISKY_EXT:
        findings.append({"id": "DOUBLE_EXTENSION", "sev": 25,
                          "detail": f"{filename} — inner extension is {stem_ext}"})

    return findings

### 5.2 HTML Smuggling

An HTML attachment that uses the browser's own APIs to assemble and force-download a payload. Because the payload is encoded inside the HTML (typically base64), nothing malicious crosses the mail gateway — only a benign-looking HTML file.

The assembly signature requires a combination of:

- `atob()` — base64 decode
- `new Blob()` — construct a binary object in memory
- `createObjectURL` or `msSaveOrOpenBlob` — create a downloadable reference
- `Uint8Array` or `download=` — write bytes / trigger the download

Three or more of these in a single HTML attachment is the smuggling signature. The `download=` attribute often reveals the payload filename — that's **`SMUGGLED_FILENAME`** (+15).

Sample 06 drops an `.iso` — container formats strip Mark-of-the-Web, so the payload inside runs without a SmartScreen prompt.

In [23]:
HTML_SMUGGLING_MARKERS = (
    "atob(", "Blob(", "createObjectURL",
    "download=", "a.download", "Uint8Array", "msSaveOrOpenBlob"
)

def check_html_smuggling(filename: str, data: bytes) -> list[dict]:
    """Detect browser-side payload assembly in HTML attachments."""
    findings = []
    ext = os.path.splitext(filename)[1].lower()

    if ext not in (".html", ".htm", ".shtml"):
        return findings

    findings.append({"id": "HTML_ATTACHMENT", "sev": 20,
                      "detail": f"{filename} — HTML attachments are rare in legitimate mail"})

    text = data.decode("utf-8", "replace")
    hits = [m for m in HTML_SMUGGLING_MARKERS if m in text]

    if len(hits) >= 3:
        findings.append({"id": "HTML_SMUGGLING", "sev": 40,
                          "detail": "browser-side payload assembly: " + ", ".join(hits)})

    # Reveal the payload filename if present
    m = re.search(r"download\s*=\s*'([^']+)'", text)
    if m:
        findings.append({"id": "SMUGGLED_FILENAME", "sev": 15,
                          "detail": f"drops {m.group(1)}"})

    return findings

### 5.3 Password-Protected Archives

A password-protected archive cannot be scanned by the mail gateway. When the attacker supplies the password in the same message body, the protection serves one purpose only: defeating AV inspection.

The regex looks for patterns like:

    "Password for the file: Inv0ice@2024"
    "passcode is: Str0ng#Pass"

If the password is delivered out-of-band (a separate SMS or phone call), this rule does not fire — no false positive on legitimate encrypted HR payroll flows.

Sample 05 pairs this with a double-extension `.pdf.lnk` inside the archive — two escalation paths combining into a high score.

In [24]:
def check_archive(filename: str, data: bytes, body_text: str) -> list[dict]:
    """Check archives for in-body password disclosure and executable content."""
    findings = []
    ext = os.path.splitext(filename)[1].lower()

    if ext not in ARCHIVE_EXT:
        return findings

    # Password supplied in the same message body
    if re.search(r"(?i)(password|passcode)\s*(for|is|:)\s*\S+", body_text):
        findings.append({
            "id": "PASSWORD_PROTECTED_ARCHIVE", "sev": 30,
            "detail": "archive password supplied in the body — defeats gateway AV"
        })

    # Executable filenames visible in the ZIP central directory
    if ext == ".zip":
        if any(sig in data for sig in (b".lnk", b".exe", b".scr")):
            findings.append({
                "id": "ARCHIVE_CONTAINS_EXECUTABLE", "sev": 30,
                "detail": "archive listing references an executable or shortcut"
            })

    return findings

### 5.4 Assembling the Attachment Module

`attachment_findings()` walks every MIME part, skips non-attachment parts, and runs all three checks against each file. It also returns an inventory of attachments with SHA-256 and MD5 hashes for the IOC output.

In [25]:
def attachment_findings(msg) -> tuple[list[dict], list[dict]]:
    """Run all attachment checks. Returns (findings, inventory)."""
    findings, inventory = [], []

    # Get the full body text once for archive password check
    b = bodies(msg)
    body_text = b["text"] + " " + b["html"]

    for part in msg.walk():
        filename = part.get_filename()
        if not filename:
            continue

        data = part.get_payload(decode=True) or b""
        rec  = {
            "filename":     filename,
            "content_type": part.get_content_type(),
            "size":         len(data),
            "sha256":       hashlib.sha256(data).hexdigest(),
            "md5":          hashlib.md5(data).hexdigest()
        }
        inventory.append(rec)

        findings += check_extensions(filename)
        findings += check_html_smuggling(filename, data)
        findings += check_archive(filename, data, body_text)

    return findings, inventory

---
## 6. Scoring & Verdicts

The three modules each return a list of findings. This module aggregates them into a single score between 0 and 100, applies mitigating signals, and emits one of three verdicts.

Two design decisions worth understanding:

- **Mitigating signals are capped.** When the worst single finding scores ≥ 25, the total negative contribution is capped at −10. A mass of small positive signals cannot neutralise a strong malicious indicator.
- **The asymmetry is intentional.** A false negative (missed phish) costs an incident. A false positive costs an analyst 10 minutes. The thresholds and weights are tuned to reflect that asymmetry.

### 6.1 Mitigating Signals

Four conditions reduce the score. They represent genuine indicators of legitimacy, not absence of malice:

| Signal | Score | Condition |
|---|---|---|
| `INTERNAL_AUTHENTICATED` | −25 | `compauth=pass`, From domain is corporate, single Received hop |
| `LIST_UNSUB` | −8 | Has a `List-Unsubscribe` header — consistent with legitimate bulk mail |
| `KNOWN_PARTNER` | −6 | From domain is in `TRUSTED_PARTNERS` |
| `FULL_AUTH_PASS` | −6 | SPF, DKIM, and DMARC all pass |

Note that `KNOWN_PARTNER` and `FULL_AUTH_PASS` are deliberately weak. Sample 09 (thread hijacking) triggers both — a compromised supplier mailbox passes all auth and comes from a trusted domain. Mitigating signals reduce noise; they are not evidence of safety.

In [27]:
def apply_mitigating_signals(findings: list[dict], meta: dict) -> list[dict]:
    """Evaluate and append mitigating signals to the findings list."""

    # Genuine internal mail — single hop, corp domain, composite auth pass
    if (meta["auth"]["compauth"] == "pass"
            and meta["from_domain"] in CORP_DOMAINS
            and len(meta["hops"]) <= 1):
        findings.append({"id": "INTERNAL_AUTHENTICATED", "sev": -25,
                          "detail": "mitigating signal"})

    # List-Unsubscribe — consistent with legitimate bulk/marketing mail
    if meta["list_unsub"]:
        findings.append({"id": "LIST_UNSUB", "sev": -8,
                          "detail": "mitigating signal"})

    # Known trusted partner
    if registrable(meta.get("from_domain") or "") in TRUSTED_PARTNERS:
        findings.append({"id": "KNOWN_PARTNER", "sev": -6,
                          "detail": "mitigating signal"})

    # Full authentication pass
    if all(meta["auth"][k] == "pass" for k in ("spf", "dkim", "dmarc")):
        findings.append({"id": "FULL_AUTH_PASS", "sev": -6,
                          "detail": "mitigating signal"})

    return findings


def score(findings: list[dict]) -> int:
    """
    Aggregate findings into a score between 0 and 100.

    Mitigating signals are capped at -10 when any single finding scores >= 25
    to prevent strong malicious indicators being neutralised.
    """
    aggravating = sum(f["sev"] for f in findings if f["sev"] > 0)
    mitigating  = sum(f["sev"] for f in findings if f["sev"] < 0)
    worst       = max((f["sev"] for f in findings), default=0)

    if worst >= 25:
        mitigating = max(mitigating, -10)

    return max(0, min(100, aggravating + mitigating))


def verdict(score: int) -> str:
    if score >= 70:
        return "malicious"
    if score >= 40:
        return "suspicious"
    return "benign"

### 6.2 The Full Pipeline

All three modules and the scoring logic are wired together into a single `analyse()` function. Given a path to an `.eml` file it returns a structured result dict containing the verdict, score, all findings sorted by severity, extracted URLs, hop IPs, and attachment inventory.

In [28]:
def analyse(path: str) -> dict:
    """Run the full triage pipeline against a single .eml file."""
    with open(path, "rb") as fh:
        msg = email.message_from_binary_file(fh, policy=policy.default)

    # Module 1 — header forensics
    hf, meta = header_findings(msg)

    # Module 2 — URL and body analysis
    b = bodies(msg)
    uf, urls = url_findings(b, meta)

    # Module 3 — attachment triage
    af, attachments = attachment_findings(msg)

    # Combine and apply mitigating signals
    findings = hf + uf + af
    findings = apply_mitigating_signals(findings, meta)

    # Score and verdict
    total_score  = score(findings)
    final_verdict = verdict(total_score)

    # Extract external IPs from hop chain for IOC output
    hop_ips = sorted({
        ip for h in meta["hops"] for ip in h["ips"]
        if is_external_ip(ip)
    })

    return {
        "file":        os.path.basename(path),
        "score":       total_score,
        "verdict":     final_verdict,
        "subject":     meta["subject"],
        "from":        meta["from_addr"],
        "from_name":   meta["from_name"],
        "auth":        {k: meta["auth"][k] for k in ("spf","dkim","dmarc","compauth")},
        "findings":    sorted(findings, key=lambda x: -x["sev"]),
        "urls":        sorted(urls),
        "hop_ips":     hop_ips,
        "attachments": attachments,
    }

---
## 7. YARA Rules

YARA rules operate on raw bytes — kit landing pages and email attachment content. Unlike the triage pipeline which works on parsed `.eml` structure, YARA sees the file as a flat byte sequence and matches on string patterns and conditions.

Five rules were written for this corpus, covering:

1. Credential-harvesting landing pages
2. Telegram bot API exfiltration
3. Crawler/scanner evasion
4. HTML smuggling blob download
5. Brand impersonation assets

**Validation requirement:** every rule must be tested against `benign_intranet_page.html` before deployment. A rule that fires on the benign page cannot be deployed.

### 7.1 PHISH_Kit_Credential_Form_Generic

**Confidence:** Medium

Catches credential-harvesting landing pages by combining three conditions that individually have high false-positive rates but together are highly specific:

- A password input field
- A local POST handler (relative path — no external IdP involved)
- Either Microsoft sign-in field IDs (`i0116`, `i0118`) or a `victim_ip` tracking variable

The Microsoft field IDs are the strongest discriminator — they appear in kits that copied the real Microsoft sign-in page's HTML structure verbatim.

```yara
rule PHISH_Kit_Credential_Form_Generic {
    strings:
        $form_pass   = "type=\"password\""  nocase
        $post_local1 = "action=\"post.php\""   nocase
        $post_local2 = "action=\"next.php\""   nocase
        $post_local3 = "action=\"submit.php\"" nocase
        $ms_field1   = "i0116"  nocase   // Microsoft email input ID
        $ms_field2   = "i0118"  nocase   // Microsoft password input ID
        $victim      = "victim_ip" nocase

    condition:
        filesize < 500KB and
        $form_pass and
        (any of ($post_local*)) and
        (any of ($ms_field*) or $victim)
}
```

**False positive:** internal login pages that POST to a relative path. Pair with `PHISH_Kit_Telegram_Exfil` before alerting if the environment has many internal web apps.

### 7.2 PHISH_Kit_Telegram_Exfil

**Confidence:** High

Detects phishing kits that exfiltrate captured credentials over the Telegram Bot API. Common in commodity kits — low setup overhead, no attacker infrastructure to burn, and Telegram's API is rarely blocked at the perimeter.

The combination of `bot_token` + `chat_id` + Telegram API domain has essentially no legitimate use in a PHP landing page.

```yara
rule PHISH_Kit_Telegram_Exfil {
    strings:
        $bot  = "bot_token"        nocase
        $chat = "chat_id"          nocase
        $api  = "api.telegram.org" nocase
        $send = "sendMessage"      nocase

    condition:
        filesize < 500KB and
        $bot and $chat and
        ($api or $send)
}
```

**False positive:** legitimate Telegram bots embedded in custom tooling. Extremely unlikely in a mail gateway or kit corpus context.

### 7.3 PHISH_Kit_Crawler_Evasion

**Confidence:** High

Detects landing pages that check the browser's User-Agent string and redirect security crawlers away from the phishing content. A page that explicitly targets three or more scanner identities has one purpose.

The threshold of 3 crawler names reduces noise — a legitimate page might check for Googlebot to serve different content, but no legitimate page targets PhishTank, urlscan, and VirusTotal simultaneously.

```yara
rule PHISH_Kit_Crawler_Evasion {
    strings:
        $ua = "navigator.userAgent" nocase
        $b1 = "googlebot"  nocase
        $b2 = "phishtank"  nocase
        $b3 = "urlscan"    nocase
        $b4 = "virustotal" nocase
        $b5 = "netcraft"   nocase
        $b6 = "bingbot"    nocase

    condition:
        filesize < 500KB and
        $ua and
        3 of ($b*)
}
```

**False positive:** none realistic in a phishing kit or email attachment context.

### 7.4 PHISH_HTML_Smuggling_Blob_Download

**Confidence:** High

Detects HTML attachments that assemble a payload in the browser using the Blob API and force a download. The risky extension requirement (`.iso`, `.img`, `.vhd`, `.zip`) filters out legitimate HTML pages that use Blob for other purposes.

The `msSaveOrOpenBlob` alternative covers older IE/Edge implementations of the same technique — kit authors often include both for maximum browser compatibility.

```yara
rule PHISH_HTML_Smuggling_Blob_Download {
    strings:
        $decode = "atob("           nocase
        $bytes  = "Uint8Array"      nocase
        $blob   = "new Blob("       nocase
        $objurl = "createObjectURL" nocase
        $dl     = "download"        nocase
        $save   = "msSaveOrOpenBlob" nocase
        $risky1 = ".iso"  nocase
        $risky2 = ".img"  nocase
        $risky3 = ".vhd"  nocase
        $risky4 = ".zip"  nocase

    condition:
        filesize < 5MB and
        $decode and $blob and
        ($objurl or $save) and
        ($bytes or $dl) and
        any of ($risky*)
}
```

**MITRE:** T1027.006 — Obfuscated Files: HTML Smuggling

**False positive:** reporting tools that email self-contained HTML dashboards with export functionality. The risky extension requirement eliminates most of these.

### 7.5 PHISH_Kit_Brand_Impersonation_Assets

**Confidence:** Low

Detects pages that host brand assets locally alongside a password field — a hallmark of kits that were scraped directly from the real site. The `aadcdn.msftauth.net` CDN reference is particularly specific: it appears in Microsoft sign-in pages and gets copied verbatim into kits.

**Never deploy this rule in isolation.** Low confidence by design — legitimate internal SSO pages can trigger it. Always pair with `PHISH_Kit_Credential_Form_Generic` or `PHISH_Kit_Telegram_Exfil` before alerting.

```yara
rule PHISH_Kit_Brand_Impersonation_Assets {
    strings:
        $pw    = "type=\"password\"" nocase
        $logo1 = "ms_logo"    nocase
        $logo2 = "adobe_logo" nocase
        $logo3 = "okta_logo"  nocase
        $cdn   = "aadcdn.msftauth.net" nocase

    condition:
        filesize < 500KB and
        $pw and
        (any of ($logo*)) and
        $cdn
}
```

**False positive:** internal login pages that reference Microsoft CDN assets. Pair with another rule before alerting.

---
## 8. Sigma Rules

Sigma rules operate on mail gateway logs rather than raw file bytes. Where YARA works on content, Sigma works on structured fields — sender domain, authentication results, attachment metadata, URL patterns — as they appear in M365 threat management logs.

Five rules were written for this corpus. Field names follow the Sigma email/M365 taxonomy. Remap them to your own schema before deployment and validate with:

    sigma check sigma_rules.yml

All five rules are tagged to MITRE ATT&CK and marked `experimental` — review false-positive rates in your environment before moving to `stable`.

### 8.1 Inbound Mail Spoofing an Internal Sender Domain

**Level:** High | **Catches:** samples 01, 04

The most straightforward detection — three conditions that together have very low false-positive rate:

- Sender domain claims to be internal (`meridian-demo.local`)
- DMARC failed — the mail did not authenticate as that domain
- Network direction is inbound — rules out misconfigured internal relay

```yaml
title: Inbound Mail Spoofing an Internal Sender Domain
status: experimental
logsource:
    product: m365
    service: threat_management
    category: email
detection:
    internal_from:
        sender_domain|endswith: 'meridian-demo.local'
    auth_failed:
        dmarc: 'fail'
    external_path:
        network_direction: 'inbound'
    condition: internal_from and auth_failed and external_path
falsepositives:
    - Marketing platforms sending on your behalf without a DKIM key or SPF include.
      Onboard them properly rather than excluding them here.
level: high
tags:
    - attack.initial-access
    - attack.t1566.002
```

**MITRE:** T1566.002 — Phishing: Spearphishing Link

### 8.2 Consent Grant Request with High-Privilege Graph Scopes

**Level:** High | **Catches:** sample 07

The hardest detection in the corpus because the URL is genuinely Microsoft-hosted — reputation and sandbox tools will pass it. The detection surface is entirely in the parameters:

- URL path contains the OAuth authorize endpoint
- Scope parameter requests mailbox or file write access
- `redirect_uri` points outside Microsoft-owned domains

Operationally: pair with an allowlist of approved application (client) IDs. Alert on any `client_id` outside that list regardless of scope.

```yaml
title: Consent Grant Request with High-Privilege Graph Scopes
status: experimental
logsource:
    product: m365
    service: threat_management
    category: email
detection:
    oauth_url:
        url|contains: '/oauth2/v2.0/authorize'
    risky_scope:
        url|contains:
            - 'Mail.ReadWrite'
            - 'Mail.Send'
            - 'Files.ReadWrite.All'
            - 'offline_access'
    third_party_redirect:
        url|re: 'redirect_uri=https%3A%2F%2F(?!.*microsoft(online)?\.com)'
    condition: oauth_url and risky_scope and third_party_redirect
falsepositives:
    - Legitimate first-party integrations. Pair with an allowlist of approved
      application (client) IDs and alert on anything outside it.
level: high
tags:
    - attack.persistence
    - attack.t1550.001
```

**MITRE:** T1550.001 — Use Alternate Authentication Material

### 8.3 HTML Attachment Performing Browser-Side File Assembly

**Level:** High | **Catches:** sample 06

Detects the HTML smuggling pattern at the mail gateway log level. Requires all three assembly markers to be present in the attachment content — `atob(`, `new Blob(`, and `createObjectURL` — which eliminates the vast majority of legitimate HTML attachments.

```yaml
title: HTML Attachment Performing Browser-Side File Assembly
status: experimental
logsource:
    product: m365
    service: threat_management
    category: email
detection:
    html_attachment:
        attachment_extension:
            - 'html'
            - 'htm'
            - 'shtml'
    smuggling_markers:
        attachment_content|contains|all:
            - 'atob('
            - 'new Blob('
            - 'createObjectURL'
    condition: html_attachment and smuggling_markers
falsepositives:
    - Some reporting tools email self-contained HTML dashboards.
      Allowlist those senders explicitly — do not weaken the string set.
level: high
tags:
    - attack.defense-evasion
    - attack.t1027.006
```

**MITRE:** T1027.006 — Obfuscated Files: HTML Smuggling

### 8.4 Authenticated Partner Mail Linking to an Unrelated Domain

**Level:** Medium | **Catches:** sample 09

The thread-hijacking detection. Rated medium because of the file-sharing false-positive rate — suppliers legitimately use third-party platforms like SharePoint, DocuSign, or Dropbox. The allowlist built during onboarding is what separates a legitimate file-share link from a compromised mailbox.

When it fires without a known legitimate use, treat it as high-confidence BEC/VEC.

```yaml
title: Authenticated Partner Mail Linking to an Unrelated Domain
status: experimental
logsource:
    product: m365
    service: threat_management
    category: email
detection:
    partner_sender:
        sender_domain:
            - 'nordvale-supplies.com'
            - 'cloudsecvendor.com'
    authenticated:
        dmarc: 'pass'
        dkim: 'pass'
    offsite_link:
        url|re: 'https?://(?!([a-z0-9\-]+\.)*(nordvale-supplies|cloudsecvendor)\.com)'
    condition: partner_sender and authenticated and offsite_link
falsepositives:
    - Suppliers that legitimately use a third-party file-sharing or e-signature
      service. Build the per-supplier allowlist during onboarding, not during an incident.
level: medium
tags:
    - attack.initial-access
    - attack.t1566.001
```

**MITRE:** T1566.001 — Phishing: Spearphishing Attachment

### 8.5 Archive Attachment with Password Disclosed in Message Body

**Level:** High | **Catches:** sample 05

Clean false-positive story: if the password is delivered out-of-band (SMS, phone call, separate email), this rule does not fire. When it does fire, the intent is unambiguous — the encryption serves no purpose other than defeating gateway AV.

```yaml
title: Archive Attachment with Password Disclosed in Message Body
status: experimental
logsource:
    product: m365
    service: threat_management
    category: email
detection:
    archive:
        attachment_extension:
            - 'zip'
            - 'rar'
            - '7z'
    password_in_body:
        body|re: '(?i)(password|passcode)\s*(for|is|:)\s*\S+'
    condition: archive and password_in_body
falsepositives:
    - Payroll or HR providers who send encrypted archives with an out-of-band
      password. If the password is out of band, this rule will not fire.
level: high
tags:
    - attack.initial-access
    - attack.t1566.001
    - attack.t1027.002
```

**MITRE:** T1566.001 — Phishing: Spearphishing Attachment | T1027.002 — Software Packing

---
## 9. Evaluation

The pipeline is only as useful as its accuracy against real analyst verdicts. This section runs all 12 samples through `analyse()`, compares each predicted verdict against the gold labels, and computes precision, recall, and F1.

One framing point before looking at the numbers: the cost of a false negative (missed phish delivered to a user) is not the same as the cost of a false positive (legitimate mail quarantined). Tune the scoring weights with that asymmetry in mind.

In [31]:
# Run all 12 samples through the full pipeline
results = []
for filename in sorted(f for f in os.listdir(EMAILS_DIR) if f.endswith(".eml")):
    path = os.path.join(EMAILS_DIR, filename)
    result = analyse(path)
    results.append(result)
    print(f"{result['file']:<45} [{result['verdict'].upper():<10}] score={result['score']}")

01_o365_password_expiry.eml                   [MALICIOUS ] score=100
02_bec_bank_change.eml                        [SUSPICIOUS] score=44
03_aitm_docusign.eml                          [MALICIOUS ] score=91
04_quishing_mfa_reenroll.eml                  [MALICIOUS ] score=100
05_malspam_invoice_zip.eml                    [MALICIOUS ] score=92
06_html_smuggling_statement.eml               [MALICIOUS ] score=83
07_oauth_consent_grant.eml                    [MALICIOUS ] score=89
08_callback_subscription.eml                  [SUSPICIOUS] score=58
09_thread_hijack_reply.eml                    [SUSPICIOUS] score=42
10_punycode_vendor_portal.eml                 [MALICIOUS ] score=84
11_benign_vendor_newsletter.eml               [BENIGN    ] score=0
12_benign_internal_maintenance.eml            [BENIGN    ] score=0


In [32]:
def evaluate(results: list[dict], gold: dict) -> dict:
    """Compare predicted verdicts against analyst gold labels."""
    tp = fp = tn = fn = 0
    rows = []

    for r in results:
        g = gold.get(r["file"])
        if not g:
            continue

        actual    = g["verdict"]
        pred_bad  = r["verdict"] in ("malicious", "suspicious")
        act_bad   = actual == "malicious"

        if pred_bad and act_bad:       tp += 1
        elif pred_bad and not act_bad: fp += 1
        elif not pred_bad and act_bad: fn += 1
        else:                          tn += 1

        rows.append((
            r["file"], actual, r["verdict"],
            r["score"], "OK" if pred_bad == act_bad else "MISS"
        ))

    prec = tp / (tp + fp) if tp + fp else 0
    rec  = tp / (tp + fn) if tp + fn else 0
    f1   = 2 * prec * rec / (prec + rec) if prec + rec else 0

    return {
        "tp": tp, "fp": fp, "tn": tn, "fn": fn,
        "precision": round(prec, 3),
        "recall":    round(rec, 3),
        "f1":        round(f1, 3),
        "rows":      rows
    }


ev = evaluate(results, gold_labels)

print(f"{'File':<45} {'Gold':<12} {'Predicted':<12} {'Score':>6}  Result")
print("-" * 82)
for row in ev["rows"]:
    print(f"{row[0]:<45} {row[1]:<12} {row[2]:<12} {row[3]:>6}  {row[4]}")

print(f"\nTP={ev['tp']}  FP={ev['fp']}  TN={ev['tn']}  FN={ev['fn']}")
print(f"Precision={ev['precision']}  Recall={ev['recall']}  F1={ev['f1']}")

File                                          Gold         Predicted     Score  Result
----------------------------------------------------------------------------------
01_o365_password_expiry.eml                   malicious    malicious       100  OK
02_bec_bank_change.eml                        malicious    suspicious       44  OK
03_aitm_docusign.eml                          malicious    malicious        91  OK
04_quishing_mfa_reenroll.eml                  malicious    malicious       100  OK
05_malspam_invoice_zip.eml                    malicious    malicious        92  OK
06_html_smuggling_statement.eml               malicious    malicious        83  OK
07_oauth_consent_grant.eml                    malicious    malicious        89  OK
08_callback_subscription.eml                  malicious    suspicious       58  OK
09_thread_hijack_reply.eml                    malicious    suspicious       42  OK
10_punycode_vendor_portal.eml                 malicious    malicious        84  OK


---
## Conclusion

This notebook walked through a complete static analysis pipeline for phishing email triage. Starting from raw `.eml` files, three independent modules — header forensics, URL and body analysis, and attachment triage — each contribute weighted findings to a single aggregated score that drives a three-tier verdict.

A few principles worth carrying forward:

- **Authentication pass is not a trust signal.** A correctly configured lookalike domain passes SPF, DKIM, and DMARC. Auth tells you the mail is genuine from that domain — not that the domain is legitimate.
- **The mitigating cap matters.** Legitimate signals reduce noise but must never neutralise a strong malicious indicator. The asymmetry between a missed phish and a false positive is real and should be reflected in how weights are tuned.
- **Detection depth compounds.** No single finding catches everything. Spoofing, OAuth abuse, HTML smuggling, and supplier compromise each require a different detection surface. The value of the pipeline is that all of them are covered in a single pass.
- **Static analysis has limits.** Password-protected archives, QR codes, and callback-only lures are specifically designed to defeat it. Knowing where the pipeline's blind spots are is as important as knowing what it catches.

The gold label evaluation at the end is not just a score — it is a reminder that every weight and threshold is a tunable decision, and that tuning should be driven by operational cost, not just accuracy.

---

### Reading the Evaluation Metrics

| Metric | Formula | What it means in this context |
|---|---|---|
| **Precision** | TP / (TP + FP) | Of all emails flagged as malicious or suspicious, what fraction actually were? Low precision means analysts waste time on false alarms. |
| **Recall** | TP / (TP + FN) | Of all genuinely malicious emails, what fraction did the pipeline catch? Low recall means phishing reaches users. |
| **F1** | 2 × (P × R) / (P + R) | Harmonic mean of precision and recall. Useful as a single summary figure, but treat it carefully — in this domain a false negative is costlier than a false positive, so recall deserves more weight than precision when making tuning decisions. |

TP = true positive (correctly flagged) | FP = false positive (benign flagged as bad) | TN = true negative (correctly passed) | FN = false negative (malicious missed)